# Zepto Data & AI Platform — Module 1: Data Pipeline

I’m pulling book data from books.toscrape.com, cleaning it up, converting the prices to INR, loading it into a normalized SQLite database, and checking the output in both SQL and pandas.

The site is just a demo bookstore made for scraping practice, so there aren’t any logins or anti-bot protections to worry about. The main goal is to keep the scraper steady and polite.


In [17]:
import re
import time
import sqlite3
import warnings
from pathlib import Path

import requests
from bs4 import BeautifulSoup
import pandas as pd

warnings.filterwarnings("ignore", message=".*OpenSSL.*")  # harmless LibreSSL/urllib3 notice on macOS system python

## 1. Scraping

Instead of opening each book page one by one, I scrape the category listing pages and move through them page by page. Those pages already have most of the info we need, including the title, price, star rating, stock status, and category.

That keeps everything much faster and a lot simpler without dropping any useful fields.


In [18]:
BASE_URL = "http://books.toscrape.com/"
CATEGORIES = ["Mystery", "Fiction", "Fantasy", "Romance", "Sequential Art"]


def get_soup(url):
    resp = requests.get(url, timeout=15)
    resp.encoding = "utf-8"  # the site serves latin-1 headers but utf-8 bytes; without this the £ sign mojibakes
    return BeautifulSoup(resp.text, "lxml")


def discover_category_urls(categories):
    soup = get_soup(BASE_URL + "index.html")
    links = soup.select("div.side_categories ul li ul li a")
    found = {a.text.strip(): BASE_URL + a["href"] for a in links if a.text.strip() in categories}
    missing = set(categories) - found.keys()
    if missing:
        raise RuntimeError(f"categories not found on site: {missing}")
    return found


def scrape_category(name, start_url):
    rows = []
    url = start_url
    while url:
        soup = get_soup(url)
        for art in soup.select("article.product_pod"):
            title = art.h3.a["title"].strip()
            price_text = art.select_one("p.price_color").text.strip()
            rating_class = art.select_one("p.star-rating")["class"]
            star_word = [c for c in rating_class if c != "star-rating"][0]
            availability_text = art.select_one("p.instock.availability").text.strip()
            rows.append({
                "title": title,
                "price": price_text,
                "star_rating": star_word,
                "availability": availability_text,
                "category": name,
            })
        next_link = soup.select_one("li.next a")
        url = url.rsplit("/", 1)[0] + "/" + next_link["href"] if next_link else None
        time.sleep(0.2)  # sequential, mild pacing -- no need for anything fancier against a static test site
    return rows


def scrape_all(categories):
    cat_urls = discover_category_urls(categories)
    all_rows = []
    for name in categories:
        cat_rows = scrape_category(name, cat_urls[name])
        print(f"{name}: {len(cat_rows)} books")
        all_rows.extend(cat_rows)
    return pd.DataFrame(all_rows)

In [19]:
df_raw = scrape_all(CATEGORIES)
print("\ntotal rows:", len(df_raw))
df_raw.head()

Mystery: 32 books
Fiction: 65 books
Fantasy: 48 books
Romance: 35 books
Sequential Art: 75 books

total rows: 255


,title,price,star_rating,availability,category
0,Sharp Objects,£47.82,Four,In stock,Mystery
1,"In a Dark, Dark Wood",£19.63,One,In stock,Mystery
2,The Past Never Ends,£56.50,Four,In stock,Mystery
3,A Murder in Time,£16.64,One,In stock,Mystery
4,The Murder of Roger Ackroyd (Hercule Poirot #4),£44.10,Four,In stock,Mystery


## 2. Cleaning

This data is usable only after a bit of cleanup. The price is stored as a string with a currency symbol, the rating is written as a word like "Four" instead of a number, and the availability text is just plain English.

I use a practical rule for broken values. Numeric fields like price and rating get filled with the median when they are unreadable, while fields like title and stock status get dropped if they cannot be trusted. That keeps the pipeline stable without making up numbers that would just be misleading.


In [20]:
RATING_WORDS = {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}
GBP_TO_INR = 105.50  # fixed, project-defined constant -- not a live rate, see README

df = df_raw.copy()

df["price_gbp"] = df["price"].apply(
    lambda s: float(re.sub(r"[^0-9.]", "", str(s))) if re.search(r"\d", str(s)) else float("nan")
)
df["rating"] = df["star_rating"].map(RATING_WORDS)  # words not in the map (unexpected markup) become NaN

avail_lower = df["availability"].str.lower()
df["in_stock"] = pd.Series([pd.NA] * len(df), dtype="object")
df.loc[avail_lower.str.contains("in stock", na=False), "in_stock"] = True
df.loc[avail_lower.str.contains("out of stock", na=False), "in_stock"] = False

rows_before = len(df)

# drop: title or availability text we can't interpret at all
df = df[df["title"].notna() & (df["title"].str.strip() != "")]
df = df[df["in_stock"].notna()]

# impute: numeric fields fall back to the column median
if df["price_gbp"].isna().any():
    df["price_gbp"] = df["price_gbp"].fillna(df["price_gbp"].median())
if df["rating"].isna().any():
    df["rating"] = df["rating"].fillna(int(df["rating"].median()))

df["rating"] = df["rating"].astype(int)
df["in_stock"] = df["in_stock"].astype(bool)
df["price_inr"] = (df["price_gbp"] * GBP_TO_INR).round(2)

print(f"rows before cleaning: {rows_before}, after: {len(df)} (dropped {rows_before - len(df)})")
df[["title", "price_gbp", "price_inr", "rating", "in_stock", "category"]].head()

rows before cleaning: 255, after: 255 (dropped 0)


,title,price_gbp,price_inr,rating,in_stock,category
0,Sharp Objects,47.82,5045.01,4,True,Mystery
1,"In a Dark, Dark Wood",19.63,2070.96,1,True,Mystery
2,The Past Never Ends,56.50,5960.75,4,True,Mystery
3,A Murder in Time,16.64,1755.52,1,True,Mystery
4,The Murder of Roger Ackroyd (Hercule Poirot #4),44.10,4652.55,4,True,Mystery


In [21]:
# sanity check: dtypes and null counts on the cleaned columns
clean_cols = ["price_gbp", "price_inr", "rating", "in_stock", "category"]
print(df[clean_cols].dtypes)
print()
print(df[clean_cols].isna().sum())
print()
print(df["rating"].value_counts().sort_index())
print("\nrows:", len(df), "| categories:", df["category"].nunique())

price_gbp    float64
price_inr    float64
rating         int64
in_stock        bool
category      object
dtype: object

price_gbp    0
price_inr    0
rating       0
in_stock     0
category     0
dtype: int64

rating
1    53
2    41
3    63
4    50
5    48
Name: count, dtype: int64

rows: 255 | categories: 5


## 3. Currency conversion

I convert the price to INR using `price_gbp * 105.50`. I keep that rate fixed for this project instead of pulling a live exchange-rate API, because it makes the exercise consistent and easy to follow.

I also explain that choice in the README, but the short version is that this project needs a stable baseline rather than a moving market value.


In [22]:
df[["title", "price_gbp", "price_inr"]].sample(5, random_state=1)

,title,price_gbp,price_inr
182,This One Summer,19.49,2056.19
34,"We Love You, Charlie Freeman",50.27,5303.48
110,Island of Dragons (Unwanteds #7),29.65,3128.08
112,City of Glass (The Mortal Instruments #3),56.02,5910.11
226,"Fruits Basket, Vol. 4 (Fruits Basket #4)",50.44,5321.42


## 4. Normalized SQLite schema

I store the data in two tables: one for categories and one for books. Each book points to its category with a category_id, which keeps the database normalized and avoids repeating the category name in every row.

This is a simple one-to-many setup, and it makes the joins nice and straightforward.


In [23]:
DB_PATH = Path("books.db")
if DB_PATH.exists():
    DB_PATH.unlink()  # rebuild from scratch each run so the notebook is idempotent

cats = sorted(df["category"].unique())
categories_df = pd.DataFrame({"category_id": range(1, len(cats) + 1), "category_name": cats})
cat_id_map = dict(zip(categories_df["category_name"], categories_df["category_id"]))

books_df = df[["title", "price_gbp", "price_inr", "rating", "in_stock", "category"]].copy()
books_df["category_id"] = books_df["category"].map(cat_id_map)
books_df = books_df.drop(columns=["category"]).reset_index(drop=True)
books_df.insert(0, "book_id", range(1, len(books_df) + 1))
books_df["in_stock"] = books_df["in_stock"].astype(int)  # sqlite has no native bool, store as 0/1

conn = sqlite3.connect(DB_PATH)
cur = conn.cursor()
cur.executescript("""
CREATE TABLE categories (
    category_id   INTEGER PRIMARY KEY,
    category_name TEXT UNIQUE NOT NULL
);

CREATE TABLE books (
    book_id     INTEGER PRIMARY KEY,
    title       TEXT NOT NULL,
    price_gbp   REAL NOT NULL,
    price_inr   REAL NOT NULL,
    rating      INTEGER NOT NULL,
    in_stock    INTEGER NOT NULL,
    category_id INTEGER NOT NULL REFERENCES categories(category_id)
);
""")
categories_df.to_sql("categories", conn, if_exists="append", index=False)
books_df.to_sql("books", conn, if_exists="append", index=False)
conn.commit()

print("categories:", len(categories_df), "| books:", len(books_df))
pd.read_sql("SELECT COUNT(*) AS n FROM books", conn)

categories: 5 | books: 255


,n
0,255


## 5. SQL queries

I run a few simple checks against the database to cover the main SQL patterns: filtering, ordering, limiting, distinct values, ranges, and joins between the categories and books tables.


These are not meant to be fancy queries; they’re just the checks I want to make sure the data and schema behave the way I expect.

In [24]:
# Q1 -- SELECT/WHERE + ORDER BY + LIMIT: 10 priciest 4+ star books
q1 = """
SELECT title, rating, price_gbp, price_inr
FROM books
WHERE rating >= 4
ORDER BY price_gbp DESC
LIMIT 10;
"""
q1_result = pd.read_sql(q1, conn)
q1_result

,title,rating,price_gbp,price_inr
0,Myriad (Prentor #1),4,58.75,6198.12
1,The Rose & the Dagger (The Wrath and the Dawn #2),4,58.64,6186.52
2,Digital Fortress,5,58.00,6119.00
3,The Raven Boys (The Raven Cycle #1),4,57.74,6091.57
4,The No. 1 Ladies' Detective Agency (No. 1 Ladi...,4,57.70,6087.35
5,El Deafo,5,57.62,6078.91
6,Kitchens of the Great Midwest,5,57.20,6034.60
7,"Ajin: Demi-Human, Volume 1 (Ajin: Demi-Human #1)",4,57.06,6019.83
8,"Giant Days, Vol. 1 (Giant Days #1-4)",4,56.76,5988.18
9,The Past Never Ends,4,56.50,5960.75


In [25]:
# Q2 -- DISTINCT: every category we loaded
q2 = "SELECT DISTINCT category_name FROM categories ORDER BY category_name;"
q2_result = pd.read_sql(q2, conn)
q2_result

,category_name
0,Fantasy
1,Fiction
2,Mystery
3,Romance
4,Sequential Art


In [26]:
# Q3 -- BETWEEN: mid-priced books (GBP 20-40)
q3 = """
SELECT title, price_gbp, rating
FROM books
WHERE price_gbp BETWEEN 20 AND 40
ORDER BY price_gbp;
"""
q3_result = pd.read_sql(q3, conn)
print("rows:", len(q3_result))
q3_result.head(10)

rows: 93


,title,price_gbp,rating
0,Blood Defense (Samantha Brinkman #1),20.30,3
1,Delivering the Truth (Quaker Midwife Mystery #1),20.89,4
2,"Sit, Stay, Love",20.90,3
3,"Fruits Basket, Vol. 6 (Fruits Basket #6)",20.96,4
4,Tuesday Nights in 1980,21.04,2
5,"Saga, Volume 3 (Saga (Collected Editions) #3)",21.57,5
6,"Paper Girls, Vol. 1 (Paper Girls #1-5)",21.71,4
7,Shadow Rites (Jane Yellowrock #10),21.72,4
8,Hystopia: A Novel,21.96,4
9,Fifty Shades Darker (Fifty Shades #2),21.96,1


In [27]:
# Q4 -- IN: books from a specific subset of categories, cheapest first
target_categories = ["Mystery", "Fantasy"]
placeholders = ",".join("?" for _ in target_categories)
q4 = f"""
SELECT b.title, c.category_name, b.price_gbp
FROM books b
JOIN categories c ON b.category_id = c.category_id
WHERE c.category_name IN ({placeholders})
ORDER BY b.price_gbp ASC
LIMIT 10;
"""
q4_result = pd.read_sql(q4, conn, params=target_categories)
q4_result

,title,category_name,price_gbp
0,Tastes Like Fear (DI Marnie Rome #3),Mystery,10.69
1,Hide Away (Eve Duncan #20),Mystery,11.84
2,Every Heart a Doorway (Every Heart A Doorway #1),Fantasy,12.16
3,The Girl You Lost,Mystery,12.29
4,Sister Sable (The Mad Queen #1),Fantasy,13.33
5,Princess Between Worlds (Wide-Awake Princess #5),Fantasy,13.34
6,Playing with Fire,Mystery,13.71
7,That Darkness (Gardiner and Renner #1),Mystery,13.92
8,Harry Potter and the Chamber of Secrets (Harry...,Fantasy,14.74
9,The Girl In The Ice (DCI Erika Foster #1),Mystery,15.85


In [28]:
# Q5 -- JOIN: the 3 highest-rated books per category (ties broken by lower price, then book_id)
q5 = """
SELECT category_name, title, rating, price_gbp
FROM (
    SELECT c.category_name, b.title, b.rating, b.price_gbp, b.book_id,
           ROW_NUMBER() OVER (
               PARTITION BY b.category_id
               ORDER BY b.rating DESC, b.price_gbp ASC, b.book_id ASC
           ) AS rnk
    FROM books b
    JOIN categories c ON b.category_id = c.category_id
)
WHERE rnk <= 3
ORDER BY category_name, rnk;
"""
q5_result = pd.read_sql(q5, conn)
q5_result

,category_name,title,rating,price_gbp
0,Fantasy,Every Heart a Doorway (Every Heart A Doorway #1),5,12.16
1,Fantasy,Princess Between Worlds (Wide-Awake Princess #5),5,13.34
2,Fantasy,The Star-Touched Queen,5,32.30
3,Fiction,Dear Mr. Knightley,5,11.21
4,Fiction,The Silent Wife,5,12.34
5,Fiction,Thirst,5,17.27
6,Mystery,The Girl You Lost,5,12.29
7,Mystery,The Silkworm (Cormoran Strike #2),5,23.05
8,Mystery,What Happened on Beale Street (Secrets of the ...,5,25.37
9,Romance,A Gentleman's Position (Society of Gentlemen #3),5,14.75


## 6. pandas verification: `pd.read_sql` vs `pd.merge`

I’ve already pulled a couple of query results into pandas with `pd.read_sql`, and I want to confirm the same logic still holds when I rebuild it directly in pandas.

Here, the "top 3 rated books per category" query is recreated using `pd.merge` on the in-memory DataFrames, without SQL, and then compared directly to the SQL result.


In [29]:
# already have these as DataFrames from pd.read_sql above:
print("Q1 (read_sql) shape:", q1_result.shape)
print("Q5 (read_sql) shape:", q5_result.shape)

Q1 (read_sql) shape: (10, 4)
Q5 (read_sql) shape: (15, 4)


In [30]:
# reproduce Q5 purely with pandas: merge, then rank within each category
merged = pd.merge(books_df, categories_df, on="category_id")
merged_sorted = merged.sort_values(
    ["category_id", "rating", "price_gbp", "book_id"],
    ascending=[True, False, True, True],
    kind="mergesort",  # stable sort so ties resolve the same way sqlite's ORDER BY does
)
merged_sorted["rnk"] = merged_sorted.groupby("category_id").cumcount() + 1

pandas_result = (
    merged_sorted[merged_sorted["rnk"] <= 3]
    .sort_values(["category_name", "rnk"], kind="mergesort")[["category_name", "title", "rating", "price_gbp"]]
    .reset_index(drop=True)
)
pandas_result

,category_name,title,rating,price_gbp
0,Fantasy,Every Heart a Doorway (Every Heart A Doorway #1),5,12.16
1,Fantasy,Princess Between Worlds (Wide-Awake Princess #5),5,13.34
2,Fantasy,The Star-Touched Queen,5,32.30
3,Fiction,Dear Mr. Knightley,5,11.21
4,Fiction,The Silent Wife,5,12.34
5,Fiction,Thirst,5,17.27
6,Mystery,The Girl You Lost,5,12.29
7,Mystery,The Silkworm (Cormoran Strike #2),5,23.05
8,Mystery,What Happened on Beale Street (Secrets of the ...,5,25.37
9,Romance,A Gentleman's Position (Society of Gentlemen #3),5,14.75


In [31]:
side_by_side = q5_result.rename(columns=lambda c: f"sql_{c}").join(
    pandas_result.rename(columns=lambda c: f"pandas_{c}")
)
are_equal = q5_result.reset_index(drop=True).equals(pandas_result.reset_index(drop=True))
print("pd.read_sql and pd.merge results are identical:", are_equal)
side_by_side

pd.read_sql and pd.merge results are identical: True


,sql_category_name,sql_title,sql_rating,sql_price_gbp,pandas_category_name,pandas_title,pandas_rating,pandas_price_gbp
0,Fantasy,Every Heart a Doorway (Every Heart A Doorway #1),5,12.16,Fantasy,Every Heart a Doorway (Every Heart A Doorway #1),5,12.16
1,Fantasy,Princess Between Worlds (Wide-Awake Princess #5),5,13.34,Fantasy,Princess Between Worlds (Wide-Awake Princess #5),5,13.34
2,Fantasy,The Star-Touched Queen,5,32.30,Fantasy,The Star-Touched Queen,5,32.30
3,Fiction,Dear Mr. Knightley,5,11.21,Fiction,Dear Mr. Knightley,5,11.21
4,Fiction,The Silent Wife,5,12.34,Fiction,The Silent Wife,5,12.34
5,Fiction,Thirst,5,17.27,Fiction,Thirst,5,17.27
6,Mystery,The Girl You Lost,5,12.29,Mystery,The Girl You Lost,5,12.29
7,Mystery,The Silkworm (Cormoran Strike #2),5,23.05,Mystery,The Silkworm (Cormoran Strike #2),5,23.05
8,Mystery,What Happened on Beale Street (Secrets of the ...,5,25.37,Mystery,What Happened on Beale Street (Secrets of the ...,5,25.37
9,Romance,A Gentleman's Position (Society of Gentlemen #3),5,14.75,Romance,A Gentleman's Position (Society of Gentlemen #3),5,14.75


## 7. Final sanity check

This is the last quick pass before I wrap up. I make sure the database file exists, the row counts look reasonable, and the cleaned columns have the dtypes and values I expect.


It is a simple check, but it catches the kind of small issues that are easy to miss once the pipeline has already run successfully.

In [32]:
print("books.db exists:", DB_PATH.exists(), "-", DB_PATH.stat().st_size, "bytes")
print("total book rows:", len(books_df))
print("categories:", categories_df["category_name"].tolist())
print("rating dtype:", books_df["rating"].dtype, "| range:", books_df["rating"].min(), "-", books_df["rating"].max())
print("in_stock dtype (stored as int 0/1):", books_df["in_stock"].dtype, "| unique:", books_df["in_stock"].unique())
print("price_gbp dtype:", books_df["price_gbp"].dtype)
print("price_inr dtype:", books_df["price_inr"].dtype)
print("sample check: price_inr == price_gbp * 105.50 ->",
      bool((books_df["price_inr"] == (books_df["price_gbp"] * 105.50).round(2)).all()))

conn.close()

books.db exists: True - 36864 bytes
total book rows: 255
categories: ['Fantasy', 'Fiction', 'Mystery', 'Romance', 'Sequential Art']
rating dtype: int64 | range: 1 - 5
in_stock dtype (stored as int 0/1): int64 | unique: [1]
price_gbp dtype: float64
price_inr dtype: float64
sample check: price_inr == price_gbp * 105.50 -> True
